
This is a python tools for converting data to kml format.\
Here data will be read from wind_resources_climall_$<hhhh>$.txt file. This file will be produced when "Export to ASCII format" in Wind Resources module set to True. 

After running the script successfully, there will be 3 files in the project_folder/Layout 1/report: \
wind_resources_climall_$<hhhh>$.kml , 
wind_resources_climall_$<hhhh>$.png , 
only_colorbar.png 

Open kml file using google earth. 


Requirements:\
pandas \
utm  \
simplekml \
scipy \
matplotlib 


Prepared by Mohammadreza Mohammadpour Penchah \
mohammadreza.mohammadpour@windsim.com


In [1]:
## load required packages
import utm
import pandas as pd
import os
import simplekml
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
from matplotlib.colors import Normalize
from matplotlib.transforms import Bbox


In [2]:
### User defined variables
project_folder = r"C:/Users/MohammadrezaMohammad/Documents/WindSim Projects/test/Layout 1/report" 
filename="wind_resources_climall_0080"

input_file= os.path.join(project_folder, filename+".txt")
kml_file = os.path.join(project_folder, filename+".kml")
kml_image = os.path.join(project_folder, filename+".png")

## Center lat and lon require to find UTM zone number 
center_lat= 48.743776  
center_lon= 7.402327

legend_min_spd=0.
legend_max_spd=10.


############# End of user defined variables


In [ ]:
## Define some functions
def create_kml_from_spatial_data(data_file_path, zone_number, zone_letter, output_image, output_kml, legend_min_spd, legend_max_spd):
    # Read the data
#    df = pd.read_csv(data_file_path, delim_whitespace=True, header=None, names=['Easting', 'Northing', 'Value'])
    df = pd.read_csv(data_file_path, delimiter='\t', skipinitialspace=True ) #, header=None)
    
    # Convert UTM to lat/lon
    def utm_to_latlon(easting, northing):
        lon, lat = utm.to_latlon(easting, northing, zone_number, zone_letter, strict=False)
        return lat, lon

    df['Longitude'], df['Latitude'] = zip(*df.apply(lambda row: utm_to_latlon(row['x[m]'], row['y[m]']), axis=1))

    # Interpolate the data
    buffer = 500
    ll_lon, ll_lat = utm_to_latlon(np.min(df['x[m]'])+buffer, np.min(df['y[m]'])+buffer)
    ul_lon, ul_lat = utm_to_latlon(np.min(df['x[m]'])+buffer, np.max(df['y[m]'])-buffer)
    ur_lon, ur_lat = utm_to_latlon(np.max(df['x[m]'])-buffer, np.max(df['y[m]'])-buffer)
    lr_lon, lr_lat = utm_to_latlon(np.max(df['x[m]'])-buffer, np.min(df['y[m]'])+buffer)
    dxy = df['x[m]'][1] - df['x[m]'][0]
    Nx = (np.max(df['x[m]']) - np.min(df['x[m]']) - 2*buffer)/dxy
    Ny = (np.max(df['y[m]']) - np.min(df['y[m]']) - 2*buffer)/dxy
    min_lat = max(ll_lat, lr_lat)
    max_lat = min(ul_lat, ur_lat)
    min_lon = max(ll_lon, ul_lon)
    max_lon = min(lr_lon, ur_lon)
    grid_x, grid_y = np.meshgrid(np.linspace(min_lon,max_lon,int(Nx)), np.linspace(min_lat,max_lat,int(Ny)))
    grid_z = griddata((df['Longitude'], df['Latitude']), df['V_mean[m/s]'], (grid_x, grid_y), method='linear')
    
    min_spd = legend_min_spd
    max_spd = legend_max_spd
    
    plt.imsave(output_image, grid_z, format='png', cmap='jet', vmin=min_spd, vmax=max_spd, origin='lower')
    plt.pcolormesh(grid_x, grid_y, grid_z, vmin=min_spd, vmax=max_spd, cmap='jet')

    # Create a figure and an 'invisible' axes
    title = "Mean speed"
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.axis('off')
    # Create a mappable object with the same colormap and norm
    mappable = plt.cm.ScalarMappable(norm=Normalize(np.min(min_spd), np.max(max_spd)), cmap='jet')
    # Remove the original axes and add the colorbar
    cbar = fig.colorbar(mappable, ax=ax, orientation='vertical', label=title)
    cbar.ax.tick_params(labelsize=16)
    cbar.ax.set_ylabel(title, fontsize=16)
    # Save the plot as a PNG file
    bbox = Bbox([[4.0,0.5],[6.0,5.5]])
    colorbar_fig = os.path.join(os.path.dirname(data_file_path), "only_colorbar.png")
    plt.savefig(colorbar_fig, bbox_inches=bbox)

    # Create a KML file
    kml = simplekml.Kml()
    ground = kml.newgroundoverlay(name='Spatial Data Overlay')
    ground.icon.href = output_image
    ground.latlonbox.north = max_lat
    ground.latlonbox.south = min_lat
    ground.latlonbox.east = max_lon
    ground.latlonbox.west = min_lon
    ground.color = simplekml.Color.changealphaint(127, simplekml.Color.white)
    screen = kml.newscreenoverlay(name='ColorbarOverlay')
    screen.icon.href = colorbar_fig  # Path to your colorbar image
    screen.overlayxy = simplekml.OverlayXY(x=0, y=1, xunits=simplekml.Units.fraction, yunits=simplekml.Units.fraction)
    screen.screenxy = simplekml.ScreenXY(x=0.01, y=0.99, xunits=simplekml.Units.fraction, yunits=simplekml.Units.fraction)
    screen.size.x = 0  # Set the x size of the colorbar
    screen.size.y = 0  # Set the y size of the colorbar
    screen.size.xunits = simplekml.Units.fraction  # or pixels
    screen.size.yunits = simplekml.Units.fraction  # or pixels
    kml.save(output_kml)


def find_utm_spatial_ref(lon, lat):
    if lat < -80 or lat > 84:
        raise ValueError("Latitude out of UTM bounds")
    
    zone_number = int((lon + 180) / 6) + 1
    zone_letter = 'CDEFGHJKLMNPQRSTUVWX'[int((lat + 80) / 8)]
#    if lat >= 0:
    return zone_number, zone_letter
    

In [ ]:
zone_number, zone_letter = find_utm_spatial_ref(center_lon, center_lat)
print("zone_number, zone_letter ", zone_number,', ', zone_letter)

create_kml_from_spatial_data(input_file, zone_number, zone_letter, kml_image, kml_file, legend_min_spd, legend_max_spd)